<a href="https://colab.research.google.com/github/Ishwaryaa6369Sivakumar/cdwmain/blob/day5/Day5_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import GPT2Tokenizer, TextDataset, DataCollatorForLanguageModeling, GPT2LMHeadModel, pipeline, \
                         Trainer, TrainingArguments

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')  # load up a standard gpt2 model

tokenizer.pad_token = tokenizer.eos_token
# set our pad token to be the eos token. This lets gpt know how to fill space

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [3]:
# load up our data into a dataset
pds_data = TextDataset(
    tokenizer=tokenizer,
    file_path='/content/book.txt',  # Principles of Data Science - Sinan Ozdemir
    block_size=64  # length of each chunk of text to use as a datapoint
)

/usr/local/lib/python3.11/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


In [4]:
pds_data[0], pds_data[0].shape  # inspect the first point

(tensor([   44,  7156,  1404, 25994,   198,    51, 25994,    45, 43781,  3268,
           362,    46,    20,    46,   198, 42131,   416,   198,   198,    35,
         43664,  3698,  8782, 15154, 34509,   198,   198, 30650,   198,   198,
         42672, 40340,    13,   521,    67,   513,   198,   198,  2919,    14,
          1065,    14,  5304,  1478,    25,  2231,   628,   200, 24492,   739,
          8568, 17098,   422,   383, 36619,   416,   198, 37046, 13661, 12052,
           198,    18,  6479,  3841]),
 torch.Size([64]))

In [5]:
print(tokenizer.decode(pds_data[0]))

MEGATECH
TECHNOLOGY IN 2O5O
edited by

DANIEL FRANKLIN

Books

Megatech.indd 3

08/12/2016 14:45

Published under exclusive licence from The Economist by
Profile Books Ltd
3 Holford


In [6]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False,
    # MLM is Masked Language Modelling (for BERT + auto-encoding tasks)
)

In [7]:
# example of how collator pads data dynamically
collator_example = data_collator([tokenizer('I am an input'), tokenizer('So am I')])

collator_example

{'input_ids': tensor([[   40,   716,   281,  5128],
        [ 2396,   716,   314, 50256]]), 'attention_mask': tensor([[1, 1, 1, 1],
        [1, 1, 1, 0]]), 'labels': tensor([[  40,  716,  281, 5128],
        [2396,  716,  314, -100]])}

In [8]:
collator_example.input_ids  # 50256 is our pad token id
tokenizer.pad_token_id

50256

In [9]:
collator_example.attention_mask  # Note the 0 in the attention mask where we have a pad token

tensor([[1, 1, 1, 1],
        [1, 1, 1, 0]])

In [10]:
collator_example.labels  # note the -100 to ignore loss calculation for the padded token
# Labels are shifted inside the GPT model so we don't need to worry about that

tensor([[  40,  716,  281, 5128],
        [2396,  716,  314, -100]])

In [11]:
model = GPT2LMHeadModel.from_pretrained('gpt2')  # load up a GPT2 model

pretrained_generator = pipeline(  # create a generator with built in params
    'text-generation', model=model, tokenizer='gpt2',
    config={'max_length': 200, 'do_sample': True, 'top_p': 0.9, 'temperature': 0.7, 'top_k': 10}
)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


In [12]:
print('----------')
for generated_sequence in pretrained_generator('This dataset shows the relationship', num_return_sequences=3):
    print(generated_sequence['generated_text'])
    print('----------')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------
This dataset shows the relationship between BMI, mortality, and suicide in women at age 15–19 during a 20–year period.

All of the women on this sample who received an anonymous questionnaire were contacted through telephone surveys. Women who had completed
----------
This dataset shows the relationship between the rates of depression and physical activity, as well as by height and weight, among subjects participating in the study. Data from the individual data sets is shown in Figure 7. (d) Comparison of the rates of depression
----------
This dataset shows the relationship between IQ levels at age 18 and gender in Britain. The IQ ranges between 3.1 and 4.2 for men and between 6.8 and 10.5 for women. These IQ differences were statistically significant. The average
----------


In [13]:
training_args = TrainingArguments(
    output_dir="./gpt2_pds", #The output directory
    overwrite_output_dir=True, #overwrite the content of the output directory
    num_train_epochs=3, # number of training epochs
    per_device_train_batch_size=32, # batch size for training
    per_device_eval_batch_size=32,  # batch size for evaluation
    logging_steps=10,
    load_best_model_at_end=True,
    evaluation_strategy='epoch',
    save_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=pds_data.examples[:int(len(pds_data.examples)*.8)],
    eval_dataset=pds_data.examples[int(len(pds_data.examples)*.8):]
)

trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'eval_loss': 4.488680839538574,
 'eval_model_preparation_time': 0.0044,
 'eval_runtime': 21.3621,
 'eval_samples_per_second': 1.217,
 'eval_steps_per_second': 0.047}

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,3.968448,0.004400
2,No log,3.875020,0.004400
3,4.549000,3.849276,0.004400


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=12, training_loss=4.483018239339192, metrics={'train_runtime': 575.3746, 'train_samples_per_second': 0.532, 'train_steps_per_second': 0.021, 'total_flos': 9994420224000.0, 'train_loss': 4.483018239339192, 'epoch': 3.0})

In [15]:
trainer.evaluate()  # loss decrease is slowing down so we are hitting our limit

{'eval_loss': 3.849275827407837,
 'eval_model_preparation_time': 0.0044,
 'eval_runtime': 15.3552,
 'eval_samples_per_second': 1.693,
 'eval_steps_per_second': 0.065,
 'epoch': 3.0}

In [16]:
trainer.save_model()

In [17]:
loaded_model = GPT2LMHeadModel.from_pretrained('./gpt2_pds')

finetuned_generator = pipeline(
    'text-generation', model=loaded_model, tokenizer=tokenizer,
    config={'max_length': 200, 'do_sample': True, 'top_p': 0.9, 'temperature': 0.7, 'top_k': 10}
)

Device set to use cpu


In [20]:
# examples are now sustainably about data
print('----------')
for generated_sequence in finetuned_generator('How do the "edge cases" of today, such as the early adoption of smartphones in Japan or mobile money in Kenya, inform predictions about the technologies of 2050', num_return_sequences=3):
    print(generated_sequence['generated_text'])
    print('----------')

----------
How do the "edge cases" of today, such as the early adoption of smartphones in Japan or mobile money in Kenya, inform predictions about the technologies of 2050?

One aspect of the challenges is that technological change makes it necessary for "many
----------
How do the "edge cases" of today, such as the early adoption of smartphones in Japan or mobile money in Kenya, inform predictions about the technologies of 2050?

I have argued for decades that many of the world's leading AI startups and
----------
How do the "edge cases" of today, such as the early adoption of smartphones in Japan or mobile money in Kenya, inform predictions about the technologies of 2050? The new technologies and technologies also threaten the privacy that can be accrued in digital-age
----------
